# PINGMapper → VForce **Flow** + **Lakehouse**

This notebook is the reference example for how **VForce Flow** (orchestration) and **VForce Lakehouse** (storage + catalog + analytics) work together, using
[PINGMapper](https://cameronbodine.github.io/PINGMapper/) — an open-source tool that turns consumer side-scan **sonar** recordings into georeferenced imagery and benthic-substrate maps.

**The pattern**
1. **Process** — run PINGMapper on a sonar recording → GeoTIFFs, sonograms, substrate maps, mosaics.
2. **Store** — push that imagery into **VForce Lakehouse** so it's an object + a catalog row you can search, view, and query.
3. **Orchestrate** — in production you don't run this by hand: a **VForce Flow** pipeline does step 1 → step 2 on every new recording, and every execution shows up in the Flow **Runs** view. The flow lives as code in a **git-sourced project**.


## 0 · Setup

PINGMapper pulls heavy geospatial deps (GDAL, rasterio). The cleanest install is the project's Miniforge/`pixi` environment (see the [docs](https://cameronbodine.github.io/PINGMapper/)); for a notebook, `pip` works if your kernel already has GDAL:


In [ ]:
# Recommended: a conda/Miniforge env per the PINGMapper docs.
# In a ready kernel, this is enough:
%pip install -q pingmapper requests


## 1 · Process a sonar recording with PINGMapper

Fastest path — the built-in test downloads a small sample recording and runs the whole pipeline:


In [ ]:
# One-liner (downloads the small sample dataset, ~ a few MB, and processes it):
# !python -m pingmapper test 1
#
# Below we do the same thing *scripted*, so we control the output folder and can
# hand the results to the Lakehouse in step 2.


In [ ]:
import os, sys, time, glob, zipfile, requests

from pingmapper.main_readFiles import read_master_func
from pingmapper.main_rectify import rectify_master_func
from pingmapper.main_mapSubstrate import map_master_func
from pingmapper.funcs_model import DEPTH_DETECTION_AVAILABLE

WORK = os.path.abspath('./pingmapper_work')
DATA = os.path.join(WORK, 'data')
os.makedirs(DATA, exist_ok=True)

# --- download the small sample sonar recording (Humminbird .DAT + .SON) ---
ds_name = 'Test-Small-DS'
ds_path = os.path.join(DATA, ds_name)
if not os.path.exists(ds_path + '.DAT'):
    url = 'https://github.com/CameronBodine/PINGMapper/releases/download/data/%s.zip' % ds_name
    print('Downloading sample recording…')
    zf = ds_path + '.zip'
    with open(zf, 'wb') as f:
        f.write(requests.get(url, allow_redirects=True).content)
    with zipfile.ZipFile(zf) as z:
        z.extractall(DATA)
    os.remove(zf)
print('Recording:', ds_path + '.DAT')


### Parameters
PINGMapper is driven by one parameter dict. These are the documented defaults from `test_PINGMapper.py`; tune freely (resolution, EGN gain, depth detection, substrate mapping, mosaics, …).


In [ ]:
inFile  = os.path.abspath(ds_path + '.DAT')
sonPath = os.path.abspath(ds_path)
projDir = os.path.abspath(os.path.join(WORK, 'PINGMapper-' + ds_name))
sonFiles = sorted(glob.glob(os.path.join(sonPath, '*.SON')))

# PINGMapper copies the 'script' into the project for reproducibility — give it a real file.
script_src = os.path.abspath('PINGMapper_lakehouse_flow.ipynb')
if not os.path.exists(script_src):
    script_src = os.path.abspath('pingmapper_run.py'); open(script_src, 'a').close()
copied = 'PINGMapper_run_' + time.strftime('%Y-%m-%d_%H%M') + '.py'

params = {
    'logfilename': os.path.join(projDir, 'log.txt'),
    'project_mode': 1,            # 1 = overwrite if it exists
    'script': [script_src, copied],
    'inFile': inFile, 'sonFiles': sonFiles, 'projDir': projDir,
    'tempC': 10, 'nchunk': 500, 'exportUnknown': True, 'fixNoDat': False, 'threadCnt': 0.5,
    'pix_res_son': 0.05, 'pix_res_map': 0.25,
    'x_offset': 0.0, 'y_offset': 0.0,
    'egn': True, 'egn_stretch': 1, 'egn_stretch_factor': 0.5, 'tone_gamma': 1.0, 'tone_gain': 1.0,
    'tileFile': '.jpg', 'wcp': True, 'wcr': True, 'wco': True, 'wcm': True,
    'sonogram_colorMap': 'copper', 'spdCor': True, 'maxCrop': True, 'USE_GPU': False,
    'remShadow': 1, 'detectDep': 1, 'smthDep': True, 'adjDep': 0, 'pltBedPick': True,
    'rect_wcp': True, 'rect_wcr': True, 'son_colorMap': 'Greys_r',
    'pred_sub': 1, 'map_sub': 1, 'export_poly': True, 'map_predict': False,
    'pltSubClass': True, 'map_class_method': 'max',
    'mosaic_nchunk': 0, 'mosaic': 1, 'map_mosaic': 1,
    'banklines': True, 'coverage': True,
}

# Substrate prediction needs the ML deps; disable gracefully if absent.
if not DEPTH_DETECTION_AVAILABLE:
    params.update(remShadow=0, detectDep=0, pred_sub=0, pltSubClass=False,
                  map_sub=0, export_poly=False, map_predict=False)
print('%d SON channels · output → %s' % (len(sonFiles), projDir))


### Run the pipeline
`read → rectify → map` — the same three master functions the CLI calls.


In [ ]:
read_master_func(**params)                       # decode pings → sonograms
if params['rect_wcp'] or params['rect_wcr'] or params['banklines'] or params['coverage']:
    rectify_master_func(**params)                # georeference → GeoTIFF mosaics, tracklines
if params['pred_sub'] or params['map_sub'] or params['export_poly']:
    map_master_func(**params)                    # ML substrate classification → maps
print('Done. Outputs in', projDir)


### What got produced


In [ ]:
outs = []
for ext in ('*.tif', '*.jpg', '*.shp', '*.png'):
    outs += glob.glob(os.path.join(projDir, '**', ext), recursive=True)
for p in sorted(outs)[:40]:
    print(round(os.path.getsize(p)/1024), 'KB ', os.path.relpath(p, projDir))
print('\n%d output files total' % len(outs))


## 2 · Store the imagery in VForce Lakehouse

Zip the georeferenced outputs and ingest them as a **batch** — the Lakehouse persists each as an object in open storage (MinIO/Iceberg) and a catalog row, so the imagery is searchable, viewable in the doc viewer, and queryable in SQL.

> Set `LAKEHOUSE_URL` / `LAKEHOUSE_TOKEN` for your environment. The platform is fronted by oauth2 — pass a bearer token (or run inside the cluster).


In [ ]:
LAKEHOUSE_URL   = os.environ.get('LAKEHOUSE_URL', 'https://lakehouse.vforce360.ai')
LAKEHOUSE_TOKEN = os.environ.get('LAKEHOUSE_TOKEN', '')          # bearer token
TENANT          = os.environ.get('LAKEHOUSE_TENANT', 'default')
BATCH_ID        = 'sonar-' + ds_name + '-' + time.strftime('%Y%m%d-%H%M')

# Bundle the GeoTIFFs + sonograms into one archive.
archive = os.path.join(WORK, BATCH_ID + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in outs:
        if p.lower().endswith(('.tif', '.jpg', '.png')):
            z.write(p, arcname=os.path.relpath(p, projDir))
print('Archive:', archive, '(%d KB)' % round(os.path.getsize(archive)/1024))

headers = {'X-Tenant-Id': TENANT}
if LAKEHOUSE_TOKEN:
    headers['Authorization'] = 'Bearer ' + LAKEHOUSE_TOKEN

with open(archive, 'rb') as f:
    resp = requests.post(LAKEHOUSE_URL + '/api/v1/ingest/archive',
                         headers=headers,
                         files={'file': (os.path.basename(archive), f, 'application/zip')})
print(resp.status_code, resp.text[:400])


Single-object alternative — ingest one GeoTIFF directly (`POST /api/v1/ingest/document`, `kind` ∈ `tiff|jpeg|jp2|pdf`):


In [ ]:
import base64
tifs = [p for p in outs if p.lower().endswith('.tif')]
if tifs:
    with open(tifs[0], 'rb') as f:
        payload = {'batch_id': BATCH_ID, 'kind': 'tiff', 'data': base64.b64encode(f.read()).decode()}
    r = requests.post(LAKEHOUSE_URL + '/api/v1/ingest/document', headers=headers, json=payload)
    print(r.status_code, r.text[:300])   # -> {"doc_id": ..., "doc_type": ...}


Confirm it landed — query the catalog over SQL (Trino):


In [ ]:
r = requests.post(LAKEHOUSE_URL + '/api/v1/compute/sql', headers=headers,
                  json={'sql': "SELECT doc_type, count(*) FROM cedms.microfiche.pages GROUP BY 1"})
print(r.status_code, r.text[:500])


## 3 · The Flow pipeline — Flow + Lakehouse together

Doing this by hand is the demo; in production a **VForce Flow** orchestrates it. The flow below triggers on a **new sonar recording**, calls a PINGMapper processing service, then hands the imagery to the **`lakehouse` connector** to store + catalog it — and every run is visible in the Flow **Runs** view.

This is **flows-as-code**: it lives in a git repo bound to a Flow **Project**, and `flowc` compiles it into a deployable object that runs autonomously and reports its runs back to the platform.


In [ ]:
flow_yaml = '''id: pingmapper-ingest
name: PINGMapper -> Lakehouse ingest
version: 1
trigger:
  kind: event              # a new sonar recording lands in object storage
  config: { source: aws-s3, subject: "sonar/uploads" }
entry: process
nodes:
  - id: process
    name: Run PINGMapper
    kind: action           # call the PINGMapper processing service (heavy Python/GDAL/ML)
    connector: http
    action: request
    inputs:
      method: POST
      url: "http://pingmapper-runner/process"
      body: { recordingKey: "{{ trigger.data.key }}" }
  - id: store
    name: Store imagery in Lakehouse
    kind: action
    connector: lakehouse
    action: ingestDocument
    inputs:
      batch_id: "{{ trigger.data.key }}"
      kind: tiff
      data: "{{ steps.process.output.geotiffBase64 }}"
  - id: done
    name: Done
    kind: terminate
    config: { status: succeeded }
edges:
  - { from: process, to: store }
  - { from: store, to: done }
'''
with open('flows/pingmapper-ingest.yaml', 'w') as f:
    os.makedirs('flows', exist_ok=True); f.write(flow_yaml)
print(flow_yaml)


Compile the flow into a deployable object with `flowc` (any CI/CD can do this):


In [ ]:
# !flowc validate flows/
# !flowc build flows/ -o dist/flowpack.tar.gz --name pingmapper
print('flowc validate flows/  →  flowc build flows/ -o dist/flowpack.tar.gz')


---
**Recap** — PINGMapper turned a raw sonar log into georeferenced habitat imagery (**process**); VForce Lakehouse stored + cataloged it (**store**); and a VForce Flow ties the two into one autonomous, git-sourced pipeline whose runs are observable in the platform (**orchestrate**). That's the Flow + Lakehouse pattern.
